<a href="https://colab.research.google.com/github/oflodahub/Practica_telegram/blob/main/Bot_para_voto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!python --version

Python 3.12.13


In [21]:
!pip install python-telegram-bot

In [22]:
!pip install nest_asyncio
import nest_asyncio
nest_asyncio.apply()

In [33]:
#!/usr/bin/env python
# pylint: disable=unused-argument
# This program is dedicated to the public domain under the CC0 license.
import re
from google.colab import drive
import pandas as pd

drive.mount('/content/gdrive')


def validar_equipo(cadena):
    # ^ : inicio de cadena
    # equipo : palabra literal
    # \s* : espacio en blanco opcional (por si es "equipo 1")
    # \d{1,2} : uno o dos dígitos
    # $ : fin de cadena
    patron = r"^equipo\s*\d{1,2}$"

    # re.IGNORECASE hace que no importe si es "Equipo" o "EQUIPO"
    if re.match(patron, cadena, re.IGNORECASE):
        return True
    else:
        return False

def es_numero(valor):
    try:
        float(valor)
        return True
    except ValueError:
        return False

"""
Simple Bot to reply to Telegram messages.

First, a few handler functions are defined. Then, those functions are passed to
the Application and registered at their respective places.
Then, the bot is started and runs until we press Ctrl-C on the command line.

Usage:
Basic Echobot example, repeats messages.
Press Ctrl-C on the command line or send a signal to the process to stop the
bot.
"""

import logging

from telegram import ForceReply, Update
from telegram.ext import Application, CommandHandler, ContextTypes, MessageHandler, filters

mensaje_error = "Debes mandar tu voto como:/voto equipoX califiacion"
lista_id = []
lista_nombre = []
lista_apellido = []
lista_equipo = []
lista_calificacion = []
lista_fecha = []
numero_votos = 0
dic_votos = {}
# Enable logging
logging.basicConfig(
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s", level=logging.INFO
)
# set higher logging level for httpx to avoid all GET and POST requests being logged
logging.getLogger("httpx").setLevel(logging.WARNING)

logger = logging.getLogger(__name__)


# Define a few command handlers. These usually take the two arguments update and
# context.
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Send a message when the command /start is issued."""
    user = update.effective_user
    await update.message.reply_html(
        rf"Hi {user.mention_html()}!",
        reply_markup=ForceReply(selective=True),
    )

def validacion():
  print("Validacion")
  pass

def parseText(upd):
  global numero_votos
  listaTexto = upd.message.text.split()
  identificador = upd.effective_chat.id
  nombre = upd.message.chat.first_name
  apellido = upd.message.chat.last_name
  equipo = listaTexto[1]
  calificacion = listaTexto[2]
  fecha = upd.message.date
  if validar_equipo(equipo) == False:
    return mensaje_error

  if es_numero(calificacion) == False:
    return mensaje_error

  print(f"DEBUG: identificador = {identificador}")
  print(f"DEBUG: nombre = {nombre}")
  print(f"DEBUG: apellido = {apellido}")
  print(f"DEBUG: equipo = {equipo}")
  print(f"DEBUG: calificacion = {calificacion}")
  print(f"DEBUG: fecha = {fecha}")
  llave = f"{identificador} {equipo}"

  lista_id.append(identificador)
  lista_nombre.append(nombre)
  lista_apellido.append(apellido)
  lista_equipo.append(equipo)
  lista_calificacion.append(calificacion)
  lista_fecha.append(fecha)
  dic_votos[llave] = numero_votos
  numero_votos = numero_votos + 1
  '''
  if llave in dic_votos:
    i = int(dic_votos[llave])
    lista_id[i]=identificador
    lista_nombre[i]=nombre
    lista_apellido[i]=apellido
    lista_equipo[i]=equipo
    lista_calificacion[i]=calificacion
    lista_fecha[i]=fecha
    return f"se reemplazo el voto número de votos {numero_votos}"
  else:
      lista_id.append(identificador)
      lista_nombre.append(nombre)
      lista_apellido.append(apellido)
      lista_equipo.append(equipo)
      lista_calificacion.append(calificacion)
      lista_fecha.append(fecha)
      dic_votos[llave] = numero_votos
      numero_votos = numero_votos + 1
'''
  return f"número de votos {numero_votos}"

async def voto_command(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Send a message when the command /voto is issued."""
    #crear id, nombre, apellido, equipo, calificación, fecha y hora
    print("Comando voto")
    print(f"DEBUG: update.callback_query = {update.callback_query}")
    respuesta = parseText(update)
    await update.message.reply_text(f"Voto recibido {update.message.chat.first_name}! {respuesta}")

async def help_command(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Send a message when the command /help is issued."""
    print("Comando help")
    print(f"DEBUG: update identificador = {update.effective_chat.id}")

    await update.message.reply_text("Help!")

async def guarda_command(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Guarda."""
    print("Comando guarda")
    print(f"DEBUG: update identificador = {update.effective_chat.id}")
    data = {"Identificador": lista_id,
            "Nombre": lista_nombre,
            "Apellido": lista_apellido,
            "Equipo": lista_equipo,
            "Calificacion": lista_calificacion,
            "Fecha": lista_fecha}

    df = pd.DataFrame(data)
    print(df)
    df.to_csv('/content/gdrive/MyDrive/infografia_cal/calificaciones.csv', index=False)
    await update.message.reply_text(f"Se guardaron {numero_votos} registros")

async def echo(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Echo the user message."""
    await update.message.reply_text(mensaje_error)

async def procesa_command(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Procesa."""
    dic_equipos = {}
    for i in range(len(lista_equipo)):
      llave = lista_equipo[i]
      if llave in dic_equipos:
          lista_cal = dic_equipos[llave]
          n = lista_cal[1]
          cal = lista_cal[0] * n

          n = n+1
          promedio = (cal + float(lista_calificacion[i])) / n
          lista_cal_nueva = [promedio, n]

          dic_equipos[llave] = lista_cal_nueva
      else:
          lista_cal = [float(lista_calificacion[i]), 1]
          dic_equipos[llave] = lista_cal
    print(dic_equipos)

    df = pd.DataFrame(dic_equipos)
    print(df)
    df.to_csv('/content/gdrive/MyDrive/infografia_cal/promedios.csv', index=False)
    await update.message.reply_text(f"Se guardaron {df} registros")

def main() -> None:
    """Start the bot."""
    # Create the Application and pass it your bot's token.
    application = Application.builder().token("8712344271:AAFvhaYLQhW4JzC4vyWntavyzqVfOLdIGWw").build()

    # on different commands - answer in Telegram
    application.add_handler(CommandHandler("start", start))
    application.add_handler(CommandHandler("help", help_command))
    application.add_handler(CommandHandler("voto", voto_command))
    application.add_handler(CommandHandler("guarda", guarda_command))
    application.add_handler(CommandHandler("procesa", procesa_command))

    # on non command i.e message - echo the message on Telegram
    application.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, echo))

    # Run the bot until the user presses Ctrl-C
    application.run_polling(allowed_updates=Update.ALL_TYPES)


if __name__ == "__main__":
    main()

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
Comando voto
DEBUG: update.callback_query = None
DEBUG: identificador = 1203501757
DEBUG: nombre = Adolfo
DEBUG: apellido = Bravo Hernández
DEBUG: equipo = equipo1
DEBUG: calificacion = 8
DEBUG: fecha = 2026-03-31 22:30:50+00:00
Comando voto
DEBUG: update.callback_query = None
DEBUG: identificador = 1203501757
DEBUG: nombre = Adolfo
DEBUG: apellido = Bravo Hernández
DEBUG: equipo = equipo2
DEBUG: calificacion = 9
DEBUG: fecha = 2026-03-31 22:31:02+00:00
Comando voto
DEBUG: update.callback_query = None
DEBUG: identificador = 1203501757
DEBUG: nombre = Adolfo
DEBUG: apellido = Bravo Hernández
DEBUG: equipo = equipo3
DEBUG: calificacion = 9
DEBUG: fecha = 2026-03-31 22:31:13+00:00
Comando voto
DEBUG: update.callback_query = None
DEBUG: identificador = 1203501757
DEBUG: nombre = Adolfo
DEBUG: apellido = Bravo Hernández
DEBUG: equipo = equipo1
DEBUG: calificacio

ERROR:telegram.ext.Application:No error handlers are registered, logging exception.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_application.py", line 1315, in process_update
    await coroutine
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_handlers/basehandler.py", line 159, in handle_update
    return await self.callback(update, context)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_18007/3318863552.py", line 182, in procesa_command
    promedio = (cal + lista_calificacion[i]) / n
               ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^~~
TypeError: unsupported operand type(s) for /: 'str' and 'int'


RuntimeError: Cannot close a running event loop